# Advanced Pandas Homework Assignment

## Overview

This homework assignment is designed to test your understanding of advanced pandas functionalities. You will work with complex data transformations, multi-index operations, performance optimization, and custom functionality extensions.

## Dataset Description

You will be working with three datasets:

1. A sales transactions dataset
2. A customer information dataset
3. A product information dataset

These datasets are designed to simulate real-world data challenges that require advanced pandas techniques to solve efficiently.

## Tasks

#0. ** Generate Data

In [1]:
import pandas as pd
import numpy as np
import datetime
from faker import Faker
import uuid

In [2]:
# Set random seed for reproducibility
np.random.seed(42)
fake = Faker()
Faker.seed(42)

In [3]:
# Define constants
num_customers = 1000
num_products = 200
num_transactions = 50000
start_date = datetime.datetime(2020, 1, 1)
end_date = datetime.datetime(2023, 12, 31)
days_range = (end_date - start_date).days

In [4]:
# Product categories and subcategories
categories = ['Electronics', 'Clothing', 'Home', 'Food', 'Beauty']
subcategories = {
    'Electronics': ['Phones', 'Computers', 'Accessories', 'TVs', 'Audio'],
    'Clothing': ['Men', 'Women', 'Children', 'Shoes', 'Accessories'],
    'Home': ['Furniture', 'Kitchen', 'Decor', 'Bedding', 'Bath'],
    'Food': ['Produce', 'Bakery', 'Dairy', 'Meat', 'Beverages'],
    'Beauty': ['Skincare', 'Makeup', 'Haircare', 'Fragrance', 'Bath & Body']
}

In [5]:
# Regions and countries
regions = ['North America', 'Europe', 'Asia', 'South America', 'Africa', 'Oceania']
countries_by_region = {
    'North America': ['USA', 'Canada', 'Mexico'],
    'Europe': ['UK', 'Germany', 'France', 'Italy', 'Spain'],
    'Asia': ['China', 'Japan', 'India', 'South Korea', 'Singapore'],
    'South America': ['Brazil', 'Argentina', 'Colombia', 'Chile', 'Peru'],
    'Africa': ['South Africa', 'Egypt', 'Nigeria', 'Kenya', 'Morocco'],
    'Oceania': ['Australia', 'New Zealand', 'Fiji']
}

In [6]:
# Generate customer data
customer_ids = [str(uuid.uuid4()) for _ in range(num_customers)]
customer_data = []

for customer_id in customer_ids:
    region = np.random.choice(regions)
    country = np.random.choice(countries_by_region[region])
    join_date = start_date + datetime.timedelta(days=np.random.randint(0, days_range))

    customer_data.append({
        'customer_id': customer_id,
        'name': fake.name(),
        'email': fake.email(),
        'phone': fake.phone_number(),
        'region': region,
        'country': country,
        'city': fake.city(),
        'join_date': join_date,
        'tier': np.random.choice(['Bronze', 'Silver', 'Gold', 'Platinum'], p=[0.5, 0.3, 0.15, 0.05]),
        'is_active': np.random.choice([True, False], p=[0.9, 0.1])
    })

customers_df = pd.DataFrame(customer_data)


In [7]:
customers_df.head

<bound method NDFrame.head of                               customer_id              name  \
0    464de727-776d-4f5e-8865-9f4957048699      Allison Hill   
1    e19606a7-0375-4868-8522-c1056b6fff35        Sean Blake   
2    4a27ad7e-d858-406d-b428-56867ab30588     Edward Fuller   
3    91dcdfe1-1dff-421d-a827-6082b861d3cd     Melinda Jones   
4    7759053e-791e-4cba-9bec-2b72ee6e5155     Charles Mcgee   
..                                    ...               ...   
995  45eb6843-550f-4615-8ff3-a708cba0a0cd  Christine Dillon   
996  c38f6456-49f3-4403-8bd3-6216ea1cbff4      Steven Mills   
997  06550ff9-07cd-4375-a26e-1231b329b369  Krystal Thompson   
998  6edbf779-5991-4a50-b3be-2fb2f3545a8a     Jasmine Scott   
999  e660bf5e-bb97-4d06-9c7f-020521bb86c8    Crystal Baxter   

                         email                phone         region  \
0     donaldgarcia@example.net      +1-219-560-0133  South America   
1    helenpeterson@example.org         651.216.1559         Europe   
2  

In [8]:
# Generate product data
product_ids = [str(uuid.uuid4()) for _ in range(num_products)]
product_data = []

for product_id in product_ids:
    category = np.random.choice(categories)
    subcategory = np.random.choice(subcategories[category])
    launch_date = start_date + datetime.timedelta(days=np.random.randint(0, days_range))

    product_data.append({
        'product_id': product_id,
        'name': fake.word() + ' ' + fake.word().capitalize(),
        'category': category,
        'subcategory': subcategory,
        'price': round(np.random.uniform(10, 1000), 2),
        'cost': round(np.random.uniform(5, 500), 2),
        'weight_kg': round(np.random.uniform(0.1, 20), 2),
        'launch_date': launch_date,
        'is_discontinued': np.random.choice([True, False], p=[0.1, 0.9])
    })

products_df = pd.DataFrame(product_data)


In [9]:
products_df.head()

,product_id,name,category,subcategory,price,cost,weight_kg,launch_date,is_discontinued
0,69aad5db-155c-4c56-8be0-aa6cb72e9eb4,since Between,Food,Produce,838.89,68.05,2.61,2022-10-01,False
1,ffe75b91-406a-4a9a-b531-2637442d2994,finally Doctor,Electronics,Phones,919.21,326.65,2.16,2023-08-26,False
2,146c5394-ebd0-4355-b859-61de36e09af7,lose Mouth,Home,Decor,23.50,164.75,11.13,2022-05-14,False
3,3f3a9e62-ef87-47cc-b147-f32418ea33e2,what Necessary,Beauty,Haircare,899.80,278.79,16.26,2020-10-17,False
4,2baabf1f-b74c-4525-9bfb-dc31a97500e5,everybody Fact,Clothing,Accessories,394.84,123.95,1.99,2022-04-16,False


In [10]:
# Generate transaction data
transaction_data = []

for _ in range(num_transactions):
    transaction_date = start_date + datetime.timedelta(days=np.random.randint(0, days_range))
    customer_id = np.random.choice(customer_ids)

    # Each transaction can have 1-5 items
    num_items = np.random.randint(1, 6)

    for _ in range(num_items):
        product_id = np.random.choice(product_ids)
        product_price = products_df.loc[products_df['product_id'] == product_id, 'price'].iloc[0]
        quantity = np.random.randint(1, 5)

        # Apply random discount
        discount_pct = np.random.choice([0, 0, 0, 0.05, 0.1, 0.15, 0.2], p=[0.6, 0.1, 0.1, 0.05, 0.05, 0.05, 0.05])
        price_after_discount = round(product_price * (1 - discount_pct), 2)

        transaction_id = str(uuid.uuid4())

        transaction_data.append({
            'transaction_id': transaction_id,
            'customer_id': customer_id,
            'product_id': product_id,
            'date': transaction_date,
            'quantity': quantity,
            'unit_price': product_price,
            'discount_pct': discount_pct,
            'price_after_discount': price_after_discount,
            'total_amount': round(quantity * price_after_discount, 2),
            'payment_method': np.random.choice(['Credit Card', 'Debit Card', 'PayPal', 'Cash', 'Bank Transfer']),
            'store_id': np.random.randint(1, 50)
        })

transactions_df = pd.DataFrame(transaction_data)


In [11]:
# Add some missing values and anomalies
transactions_df.loc[np.random.choice(transactions_df.index, size=int(len(transactions_df)*0.01)), 'unit_price'] = np.nan
transactions_df.loc[np.random.choice(transactions_df.index, size=int(len(transactions_df)*0.01)), 'discount_pct'] = np.nan
transactions_df.loc[np.random.choice(transactions_df.index, size=int(len(transactions_df)*0.005)), 'total_amount'] = -1
customers_df.loc[np.random.choice(customers_df.index, size=int(len(customers_df)*0.02)), 'email'] = np.nan
products_df.loc[np.random.choice(products_df.index, size=int(len(products_df)*0.01)), 'price'] = np.nan

# Add some duplicates
dupe_indices = np.random.choice(transactions_df.index, size=int(len(transactions_df)*0.005))
dupes = transactions_df.loc[dupe_indices].copy()
transactions_df = pd.concat([transactions_df, dupes], ignore_index=True)

In [12]:
transactions_df.head()

,transaction_id,customer_id,product_id,date,quantity,unit_price,discount_pct,price_after_discount,total_amount,payment_method,store_id
0,f678c72e-cc69-4ed9-8bb5-f2859e41cd7f,d97a3db3-7d23-443e-8913-c98c632f0738,2b18fa30-b5a2-4cf3-884b-e81da306cb44,2022-04-01,4,392.08,0.00,392.08,1568.32,PayPal,9
1,647051a4-4f98-436a-8020-4bb7e9e47429,d97a3db3-7d23-443e-8913-c98c632f0738,35d3b618-19c0-49c9-b39d-e689130b7dcd,2022-04-01,3,709.89,0.00,709.89,2129.67,Debit Card,5
2,923c57d3-7e0c-485a-8a89-831bdce48a49,5bd15348-2a9e-422b-930d-cc368293ee3e,c97b3a45-ec3c-41fd-b45a-66c7303438d8,2023-11-22,4,941.92,0.00,941.92,3767.68,Cash,12
3,5d1eea6b-c5e5-445f-892d-afa0c02ec1ca,5bd15348-2a9e-422b-930d-cc368293ee3e,bfb8acbf-08cc-4c18-b82e-0405951a052a,2023-11-22,3,995.47,0.00,995.47,2986.41,Cash,38
4,a6d21f09-cc4d-4c77-8476-864568343b83,5bd15348-2a9e-422b-930d-cc368293ee3e,32d56f11-355f-47b0-8266-5a19e0ea0e44,2023-11-22,4,557.94,0.05,530.04,2120.16,Credit Card,1


In [13]:
# Create a time-series for customer status updates
status_updates = []
for customer_id in customer_ids:
    # Generate 1-5 status updates per customer
    num_updates = np.random.randint(1, 6)
    for _ in range(num_updates):
        update_date = start_date + datetime.timedelta(days=np.random.randint(0, days_range))
        status_updates.append({
            'customer_id': customer_id,
            'update_date': update_date,
            'tier': np.random.choice(['Bronze', 'Silver', 'Gold', 'Platinum']),
            'lifetime_value': round(np.random.uniform(0, 10000), 2),
            'credit_score': np.random.randint(300, 851)
        })

status_updates_df = pd.DataFrame(status_updates)
status_updates_df.sort_values('update_date', inplace=True)

In [14]:
status_updates_df.head()

,customer_id,update_date,tier,lifetime_value,credit_score
2481,100bd023-90ca-41de-830e-331f3756dc95,2020-01-01,Platinum,1803.99,776
1183,bf10a46c-892a-49bd-b38f-a9cbfcc4af34,2020-01-01,Silver,1499.63,592
573,bdd9500c-69e1-4aca-9e16-046e7ea49c6d,2020-01-01,Bronze,2869.51,748
2045,6ee2d7f1-e067-42d6-b888-bec1571513de,2020-01-02,Bronze,2433.03,631
2107,8d1cd3d8-7ad5-43c6-b86a-14d46c5af301,2020-01-03,Platinum,4536.36,402


In [15]:
# Print sample data
print("Customers sample:")
print(customers_df.head())
print("\nProducts sample:")
print(products_df.head())
print("\nTransactions sample:")
print(transactions_df.head())
print("\nStatus Updates sample:")
print(status_updates_df.head())

Customers sample:
                            customer_id           name  \
0  464de727-776d-4f5e-8865-9f4957048699   Allison Hill   
1  e19606a7-0375-4868-8522-c1056b6fff35     Sean Blake   
2  4a27ad7e-d858-406d-b428-56867ab30588  Edward Fuller   
3  91dcdfe1-1dff-421d-a827-6082b861d3cd  Melinda Jones   
4  7759053e-791e-4cba-9bec-2b72ee6e5155  Charles Mcgee   

                       email               phone         region      country  \
0   donaldgarcia@example.net     +1-219-560-0133  South America         Peru   
1                        NaN        651.216.1559         Europe       France   
2      barbara10@example.net        441.731.6475  South America     Colombia   
3  amandasanchez@example.com  (748)535-0305x6413        Oceania  New Zealand   
4        julie69@example.com   (332)887-1012x269         Europe        Spain   

            city  join_date      tier  is_active  
0  Robinsonshire 2023-07-18    Silver       True  
1    Herrerafurt 2023-05-23    Bronze       True  

In [16]:
# Save to CSV files if needed
customers_df.to_csv('customers.csv', index=False)
products_df.to_csv('products.csv', index=False)
transactions_df.to_csv('transactions.csv', index=False)
status_updates_df.to_csv('status_updates.csv', index=False)

### Task 1: Advanced Data Transformation and Reshaping

#1. **Pivot and Melt Operations**

   - Convert the sales data from long format to wide format using `pivot` or `pivot_table`
   - Transform the data back to long format using `melt`
   - Create a pivot table that shows monthly sales totals by product category with subtotals

In [56]:
transactions_df['Month'] = transactions_df['date'].dt.to_period('M')
transactions_df_merged = pd.merge(transactions_df, products_df, on='product_id', how='left')
transactions_df_merged.head()
monthly_category_sales_with_subtotals = transactions_df_merged.pivot_table(index='Month',
                                                                    columns='category',
                                                                    values='total_amount',
                                                                    aggfunc='sum',margins=True,
                                                                    margins_name='Total')
monthly_category_sales_with_subtotals.head()

category,Beauty,Clothing,Electronics,Food,Home,Total
Month,,,,,,
2020-01,773487.93,800213.53,642884.19,961390.67,755325.90,3933302.22
2020-02,632960.54,803318.78,576248.73,853723.21,688720.04,3554971.30
2020-03,819749.87,899412.48,760638.06,938639.73,801132.02,4219572.16
2020-04,727037.46,864801.97,589828.44,935140.56,810239.19,3927047.62
2020-05,737967.11,935772.53,628004.04,951226.76,889221.77,4142192.21


In [57]:
melted_pt = pd.melt(monthly_category_sales_with_subtotals.reset_index(), id_vars=['Month'],var_name='category', value_name='total_amount')
melted_pt.head()

,Month,category,total_amount
0,2020-01,Beauty,773487.93
1,2020-02,Beauty,632960.54
2,2020-03,Beauty,819749.87
3,2020-04,Beauty,727037.46
4,2020-05,Beauty,737967.11


#2. **Multi-level Indexing**

   - Create a hierarchical index on the sales data using customer region, product category, and date
   - Perform operations on specific levels of the hierarchy using `xs`
   - Unstacking and restacking levels to reshape the data for different analyses

In [58]:
### - When working with hierarchical indices, `swaplevel` and `sortlevel` can help organize your data
transactions_product_customer_df_merged = pd.merge(transactions_df_merged, customers_df, on='customer_id', how='left')
transactions_multi_index = transactions_product_customer_df_merged.set_index(['region', 'category', 'Month'])
transactions_multi_index.head()


transaction_id  \
region        category Month                                           
Africa        Food     2022-04  f678c72e-cc69-4ed9-8bb5-f2859e41cd7f   
                       2022-04  647051a4-4f98-436a-8020-4bb7e9e47429   
South America Clothing 2023-11  923c57d3-7e0c-485a-8a89-831bdce48a49   
              Food     2023-11  5d1eea6b-c5e5-445f-892d-afa0c02ec1ca   
              Beauty   2023-11  a6d21f09-cc4d-4c77-8476-864568343b83   

                                                         customer_id  \
region        category Month                                           
Africa        Food     2022-04  d97a3db3-7d23-443e-8913-c98c632f0738   
                       2022-04  d97a3db3-7d23-443e-8913-c98c632f0738   
South America Clothing 2023-11  5bd15348-2a9e-422b-930d-cc368293ee3e   
              Food     2023-11  5bd15348-2a9e-422b-930d-cc368293ee3e   
              Beauty   2023-11  5bd15348-2a9e-422b-930d-cc368293ee3e   

                                                          product_id  \
region        category Month                                           
Africa        Food     2022-04  2b18fa30-b5a2-4cf3-884b-e81da306cb44   
                       2022-04  35d3b618-19c0-49c9-b39d-e689130b7dcd   
South America Clothing 2023-11  c97b3a45-ec3c-41fd-b45a-66c7303438d8   
              Food     2023-11  bfb8acbf-08cc-4c18-b82e-0405951a052a   
              Beauty   2023-11  32d56f11-355f-47b0-8266-5a19e0ea0e44   

                                     date  quantity  unit_price  discount_pct  \
region        category Month                                                    
Africa        Food     2022-04 2022-04-01         4      392.08          0.00   
                       2022-04 2022-04-01         3      709.89          0.00   
South America Clothing 2023-11 2023-11-22         4      941.92          0.00   
              Food     2023-11 2023-11-22         3      995.47          0.00   
              Beauty   2023-11 2023-11-22         4      557.94          0.05   

                                price_after_discount  total_amount  \
region        category Month                                         
Africa        Food     2022-04                392.08       1568.32   
                       2022-04                709.89       2129.67   
South America Clothing 2023-11                941.92       3767.68   
              Food     2023-11                995.47       2986.41   
              Beauty   2023-11                530.04       2120.16   

                               payment_method  ...  launch_date  \
region        category Month                   ...                
Africa        Food     2022-04         PayPal  ...   2023-03-20   
                       2022-04     Debit Card  ...   2022-11-16   
South America Clothing 2023-11           Cash  ...   2021-09-06   
              Food     2023-11           Cash  ...   2022-04-16   
              Beauty   2023-11    Credit Card  ...   2021-09-06   

                               is_discontinued            name_y  \
region        category Month                                       
Africa        Food     2022-04           False       Terry Estes   
                       2022-04           False       Terry Estes   
South America Clothing 2023-11           False  Phillip Stephens   
              Food     2023-11           False  Phillip Stephens   
              Beauty   2023-11           False  Phillip Stephens   

                                                email               phone  \
region        category Month                                                
Africa        Food     2022-04      dkhan@example.com          3254883586   
                       2022-04      dkhan@example.com          3254883586   
South America Clothing 2023-11  ryanadams@example.net  (596)756-1308x7612   
              Food     2023-11  ryanadams@example.net  (596)756-1308x7612   
              Beauty   2023-11  ryanadams@example.net  (596)756-1308x76

In [59]:
north_region_sales = transactions_multi_index.xs('North America', level='region').sort_index()
north_region_sales.head()

transaction_id  \
category Month                                           
Beauty   2020-01  fd02a341-3da6-4c07-bcba-fccd3f07b764   
         2020-01  abc36e91-6dff-4da5-b7a9-8db8f4ccce40   
         2020-01  6524c6cb-ac2a-49b5-b710-27cb60e67353   
         2020-01  dbaab64a-18dc-4b31-8913-9be5c7041ff4   
         2020-01  69991f00-867a-4d8c-8fcc-2968a1bb3c1d   

                                           customer_id  \
category Month                                           
Beauty   2020-01  17d6fdeb-4962-41e8-8c0f-145e60504f28   
         2020-01  da49dbd4-701b-4915-880d-c2ce68e7934d   
         2020-01  8169edf6-7974-4f69-b644-63138bc306c8   
         2020-01  6abea39e-2c76-413c-94b8-c4fabe32619f   
         2020-01  4f371d7c-6b18-434f-b9b3-b3ea19710461   

                                            product_id       date  quantity  \
category Month                                                                
Beauty   2020-01  d5a232c8-8d9f-419b-9261-fc2f9e82b923 2020-01-10         2   
         2020-01  d2e454ce-0de9-4d19-ab46-85a69b630831 2020-01-16         1   
         2020-01  553e63ed-2307-4ba2-944c-908129b3590d 2020-01-02         1   
         2020-01  d5a232c8-8d9f-419b-9261-fc2f9e82b923 2020-01-18         1   
         2020-01  9d3fff0d-d5e8-44aa-8dcc-a3beac2ccf65 2020-01-31         1   

                  unit_price  discount_pct  price_after_discount  \
category Month                                                     
Beauty   2020-01      510.87           0.0                510.87   
         2020-01      951.31           0.0                951.31   
         2020-01      100.51           0.2                 80.41   
         2020-01      510.87           0.2                408.70   
         2020-01      187.24           0.0                187.24   

                  total_amount payment_method  ...  launch_date  \
category Month                                 ...                
Beauty   2020-01       1021.74         PayPal  ...   2022-05-14   
         2020-01        951.31    Credit Card  ...   2023-07-26   
         2020-01         80.41  Bank Transfer  ...   2023-06-02   
         2020-01        408.70  Bank Transfer  ...   2022-05-14   
         2020-01        187.24    Credit Card  ...   2020-05-27   

                 is_discontinued           name_y                    email  \
category Month                                                               
Beauty   2020-01           False    Jeanette Luna       gina90@example.org   
         2020-01           False    Kenneth Smith    deborah60@example.com   
         2020-01           False      Shane Short      sonia62@example.com   
         2020-01           False  Danielle Warner  brennaneric@example.com   
         2020-01           False    Robert Butler   kanejeremy@example.net   

                                   phone  country              city  \
category Month                                                        
Beauty   2020-01   +1-812-548-2554x42867   Canada     West Margaret   
         2020-01        712.319.8819x277   Mexico    Port Marymouth   
         2020-01  001-354-902-6355x82430   Mexico        Parkerview   
         2020-01            289-338-9846      USA         Gilesview   
         2020-01     +1-506-404-8058x925   Canada  Lake Jessicafurt   

                  join_date    tier is_active  
category Month                                 
Beauty   2020-01 2022-01-14  Silver      True  
         2020-01 2022-09-06  Bronze      True  
         2020-01 2023-01-16  Bronze      True  
         2020-01 2021-09-03  Silver      True  
         2020-01 2023-02-12  Silver      True  

[5 rows x 26 columns]

In [60]:
north_region_sales2 = transactions_multi_index.xs('North America', level='region').swaplevel('category', 'Month').sort_index()
north_region_sales2.head()

transaction_id  \
Month   category                                         
2020-01 Beauty    fd02a341-3da6-4c07-bcba-fccd3f07b764   
        Beauty    abc36e91-6dff-4da5-b7a9-8db8f4ccce40   
        Beauty    6524c6cb-ac2a-49b5-b710-27cb60e67353   
        Beauty    dbaab64a-18dc-4b31-8913-9be5c7041ff4   
        Beauty    69991f00-867a-4d8c-8fcc-2968a1bb3c1d   

                                           customer_id  \
Month   category                                         
2020-01 Beauty    17d6fdeb-4962-41e8-8c0f-145e60504f28   
        Beauty    da49dbd4-701b-4915-880d-c2ce68e7934d   
        Beauty    8169edf6-7974-4f69-b644-63138bc306c8   
        Beauty    6abea39e-2c76-413c-94b8-c4fabe32619f   
        Beauty    4f371d7c-6b18-434f-b9b3-b3ea19710461   

                                            product_id       date  quantity  \
Month   category                                                              
2020-01 Beauty    d5a232c8-8d9f-419b-9261-fc2f9e82b923 2020-01-10         2   
        Beauty    d2e454ce-0de9-4d19-ab46-85a69b630831 2020-01-16         1   
        Beauty    553e63ed-2307-4ba2-944c-908129b3590d 2020-01-02         1   
        Beauty    d5a232c8-8d9f-419b-9261-fc2f9e82b923 2020-01-18         1   
        Beauty    9d3fff0d-d5e8-44aa-8dcc-a3beac2ccf65 2020-01-31         1   

                  unit_price  discount_pct  price_after_discount  \
Month   category                                                   
2020-01 Beauty        510.87           0.0                510.87   
        Beauty        951.31           0.0                951.31   
        Beauty        100.51           0.2                 80.41   
        Beauty        510.87           0.2                408.70   
        Beauty        187.24           0.0                187.24   

                  total_amount payment_method  ...  launch_date  \
Month   category                               ...                
2020-01 Beauty         1021.74         PayPal  ...   2022-05-14   
        Beauty          951.31    Credit Card  ...   2023-07-26   
        Beauty           80.41  Bank Transfer  ...   2023-06-02   
        Beauty          408.70  Bank Transfer  ...   2022-05-14   
        Beauty          187.24    Credit Card  ...   2020-05-27   

                 is_discontinued           name_y                    email  \
Month   category                                                             
2020-01 Beauty             False    Jeanette Luna       gina90@example.org   
        Beauty             False    Kenneth Smith    deborah60@example.com   
        Beauty             False      Shane Short      sonia62@example.com   
        Beauty             False  Danielle Warner  brennaneric@example.com   
        Beauty             False    Robert Butler   kanejeremy@example.net   

                                   phone  country              city  \
Month   category                                                      
2020-01 Beauty     +1-812-548-2554x42867   Canada     West Margaret   
        Beauty          712.319.8819x277   Mexico    Port Marymouth   
        Beauty    001-354-902-6355x82430   Mexico        Parkerview   
        Beauty              289-338-9846      USA         Gilesview   
        Beauty       +1-506-404-8058x925   Canada  Lake Jessicafurt   

                  join_date    tier is_active  
Month   category                               
2020-01 Beauty   2022-01-14  Silver      True  
        Beauty   2022-09-06  Bronze      True  
        Beauty   2023-01-16  Bronze      True  
        Beauty   2021-09-03  Silver      True  
        Beauty   2023-02-12  Silver      True  

[5 rows x 26 columns]

In [61]:
north_region_sales_by_category = north_region_sales.groupby('category')
north_region_sales_by_category.head()

transaction_id  \
category    Month                                           
Beauty      2020-01  fd02a341-3da6-4c07-bcba-fccd3f07b764   
            2020-01  abc36e91-6dff-4da5-b7a9-8db8f4ccce40   
            2020-01  6524c6cb-ac2a-49b5-b710-27cb60e67353   
            2020-01  dbaab64a-18dc-4b31-8913-9be5c7041ff4   
            2020-01  69991f00-867a-4d8c-8fcc-2968a1bb3c1d   
Clothing    2020-01  e0723352-48ac-4a4d-ac39-f858e2bc0fd9   
            2020-01  1a3efc19-1bc2-41cd-a870-6c7e412cf10a   
            2020-01  4f59eb23-2569-4b31-abc6-633009e526b6   
            2020-01  9eb1c7c1-cdfa-43ae-899f-93a01eafd27a   
            2020-01  bc0c9867-9498-48d9-9a30-7cbc1fff5e41   
Electronics 2020-01  76e095c8-9d2b-4cfe-9f09-8a2662120708   
            2020-01  299a797a-196a-43bd-95e4-f514ad59f4f1   
            2020-01  24e8445e-5a37-4261-836c-b273e00fe06a   
            2020-01  2f36161a-5b84-416d-9442-063fb5fda3c8   
            2020-01  fdeb8b99-4517-44f8-b20a-1333a72e051e   
Food        2020-01  782cbb2e-87dc-4e82-af02-fb2063c039ff   
            2020-01  083d3b7c-0008-4580-a498-7e896fe6ad5b   
            2020-01  730a7160-6d78-45e8-b808-9dfab7f2689e   
            2020-01  30e156e7-213b-4981-8704-5d4176251ea0   
            2020-01  cd474cf1-364a-4569-b1c9-bbd3659264ae   
Home        2020-01  2ac63d39-35a5-47a7-9de5-a59cb3cdf23f   
            2020-01  aa80de33-256e-4fae-8358-3d92948b88d7   
            2020-01  8c74b003-01a5-44ee-b1ae-82b518109622   
            2020-01  35c60a0d-a909-4e42-b295-b8e7ddb16239   
            2020-01  12730941-f429-49c6-9a05-6e0564c11803   

                                              customer_id  \
category    Month                                           
Beauty      2020-01  17d6fdeb-4962-41e8-8c0f-145e60504f28   
            2020-01  da49dbd4-701b-4915-880d-c2ce68e7934d   
            2020-01  8169edf6-7974-4f69-b644-63138bc306c8   
            2020-01  6abea39e-2c76-413c-94b8-c4fabe32619f   
            2020-01  4f371d7c-6b18-434f-b9b3-b3ea19710461   
Clothing    2020-01  6823f10b-4ea9-4bd9-b149-7d1c53967c99   
            2020-01  12f22637-1cc3-400a-be1b-f7234f1ffd45   
            2020-01  6de69102-0dd8-483c-8aed-56f8da4f616f   
            2020-01  17d6fdeb-4962-41e8-8c0f-145e60504f28   
            2020-01  17d6fdeb-4962-41e8-8c0f-145e60504f28   
Electronics 2020-01  17d6fdeb-4962-41e8-8c0f-145e60504f28   
            2020-01  84834026-7e3c-4d0b-9fc8-9e1fff498dfe   
            2020-01  8169edf6-7974-4f69-b644-63138bc306c8   
            2020-01  673ec11f-a110-4bb5-a358-875787c61d44   
            2020-01  6823f10b-4ea9-4bd9-b149-7d1c53967c99   
Food        2020-01  17d6fdeb-4962-41e8-8c0f-145e60504f28   
            2020-01  17d6fdeb-4962-41e8-8c0f-145e60504f28   
            2020-01  8169edf6-7974-4f69-b644-63138bc306c8   
            2020-01  8169edf6-7974-4f69-b644-63138bc306c8   
            2020-01  673ec11f-a110-4bb5-a358-875787c61d44   
Home        2020-01  17d6fdeb-4962-41e8-8c0f-145e60504f28   
            2020-01  c8dab3f9-98d3-4a59-ad39-c67965452241   
            2020-01  da49dbd4-701b-4915-880d-c2ce68e7934d   
            2020-01  88f5de07-37d4-40ea-b448-c23ceb736b91   
            2020-01  6823f10b-4ea9-4bd9-b149-7d1c53967c99   

                                               product_id       date  \
category    Month                                                      
Beauty      2020-01  d5a232c8-8d9f-419b-9261-fc2f9e82b923 2020-01-10   
            2020-01  d2e454ce-0de9-4d19-ab46-85a69b630831 2020-01-16   
            2020-01  553e63ed-2307-4ba2-944c-908129b3590d 2020-01-02   
            2020-01  d5a232c8-8d9f-419b-9261-fc2f9e82b923 2020-01-18   
            2020-01  9d3fff0d-d5e8-44aa-8dcc-a3beac2ccf65 2020-01-31   
Clothing    2020-01  a5d14b7b-6820-41ac-97c8-341a54e6008a 2020-01-30   
            2020-01  000d6ef7-fb5e-4415-ab74-320a26377393 2020-01-04   
            2020-01  ac1c185e-6a38-4ba8-aa1b-209fa3d3be8d 2020-01-28   
            2020-01  2bafac

In [62]:
north_clothing_sales = transactions_multi_index.xs(('North America', 'Clothing'), level=('region', 'category'))
north_clothing_sales.head()

,transaction_id,customer_id,product_id,date,quantity,unit_price,discount_pct,price_after_discount,total_amount,payment_method,...,launch_date,is_discontinued,name_y,email,phone,country,city,join_date,tier,is_active
Month,,,,,,,,,,,,,,,,,,,,,
2022-10,865bcfde-3394-4511-a0b3-a564a31bfb9b,372c7103-7611-4c44-a21b-cdb6b758b349,9050dcfc-09d2-4b51-8d7b-fbfbdd8dc7d1,2022-10-22,1,542.16,0.0,542.16,542.16,Bank Transfer,...,2020-11-10,False,Michelle Brock,mchase@example.org,001-627-202-6728x8574,USA,Garciaburgh,2020-03-03,Bronze,True
2021-08,92fed870-b041-4611-85d4-3c33266b1ad1,6fdf57d5-8443-4d30-adb9-6fb965b4af3a,a5d14b7b-6820-41ac-97c8-341a54e6008a,2021-08-17,1,341.44,0.0,341.44,341.44,Cash,...,2021-09-08,False,Willie Chavez,amy14@example.net,001-385-752-4578x71584,Mexico,Mayside,2023-12-02,Silver,True
2022-02,102794ec-982b-404c-9429-7c97213184ab,432ec545-f58c-4061-a926-e06282e17d21,f100d985-8848-43de-af8f-a1cc7f66b8cb,2022-02-23,2,539.62,0.0,539.62,1079.24,PayPal,...,2022-01-31,False,Stacy Weber,zterry@example.net,(334)610-8562x8843,USA,Katherineburgh,2022-07-07,Bronze,True
2020-08,e5c19d2e-bfd0-495f-af8a-9ecd14b0e448,ec888cac-a0fc-45a9-aa79-9cca1169fa4a,f60d3955-869f-4971-bef6-81d44c5332cf,2020-08-23,3,812.06,0.0,812.06,2436.18,Credit Card,...,2020-08-08,False,Bailey Clay,martindonald@example.com,4666489008,USA,Michaelville,2023-06-05,Silver,False
2020-05,bce6e914-7b49-4adb-9e90-ea4dee8d7683,80716d9f-4618-405d-89ef-c290bc6153cf,e8958251-dfeb-4668-b36c-28ce28048cb0,2020-05-08,3,992.21,0.0,992.21,2976.63,Bank Transfer,...,2022-12-27,False,James Davis,thomasboyle@example.org,563-274-5499x045,Mexico,South Kurtfurt,2023-05-04,Silver,True


In [63]:
NorthAm_Clothing_202001_sales = transactions_multi_index.xs(('North America', 'Clothing', '2020-01'), level=('region', 'category', 'Month')) 
NorthAm_Clothing_202001_sales.head()

transaction_id  \
region        category Month                                           
North America Clothing 2020-01  e0723352-48ac-4a4d-ac39-f858e2bc0fd9   
                       2020-01  1a3efc19-1bc2-41cd-a870-6c7e412cf10a   
                       2020-01  4f59eb23-2569-4b31-abc6-633009e526b6   
                       2020-01  9eb1c7c1-cdfa-43ae-899f-93a01eafd27a   
                       2020-01  bc0c9867-9498-48d9-9a30-7cbc1fff5e41   

                                                         customer_id  \
region        category Month                                           
North America Clothing 2020-01  6823f10b-4ea9-4bd9-b149-7d1c53967c99   
                       2020-01  12f22637-1cc3-400a-be1b-f7234f1ffd45   
                       2020-01  6de69102-0dd8-483c-8aed-56f8da4f616f   
                       2020-01  17d6fdeb-4962-41e8-8c0f-145e60504f28   
                       2020-01  17d6fdeb-4962-41e8-8c0f-145e60504f28   

                                                          product_id  \
region        category Month                                           
North America Clothing 2020-01  a5d14b7b-6820-41ac-97c8-341a54e6008a   
                       2020-01  000d6ef7-fb5e-4415-ab74-320a26377393   
                       2020-01  ac1c185e-6a38-4ba8-aa1b-209fa3d3be8d   
                       2020-01  2bafac71-d8a6-46a5-8b68-0649431f6e49   
                       2020-01  765132ba-ac81-4a85-9341-86ef245500d6   

                                     date  quantity  unit_price  discount_pct  \
region        category Month                                                    
North America Clothing 2020-01 2020-01-30         4      341.44          0.00   
                       2020-01 2020-01-04         3      814.20          0.00   
                       2020-01 2020-01-28         1      794.02          0.00   
                       2020-01 2020-01-20         4      838.06          0.00   
                       2020-01 2020-01-20         2      966.61          0.15   

                                price_after_discount  total_amount  \
region        category Month                                         
North America Clothing 2020-01                341.44       1365.76   
                       2020-01                814.20       2442.60   
                       2020-01                794.02        794.02   
                       2020-01                838.06       3352.24   
                       2020-01                821.62       1643.24   

                               payment_method  ...  launch_date  \
region        category Month                   ...                
North America Clothing 2020-01         PayPal  ...   2021-09-08   
                       2020-01  Bank Transfer  ...   2023-09-03   
                       2020-01  Bank Transfer  ...   2022-04-22   
                       2020-01  Bank Transfer  ...   2020-03-05   
                       2020-01  Bank Transfer  ...   2020-06-25   

                               is_discontinued         name_y  \
region        category Month                                    
North America Clothing 2020-01           False  William Brown   
                       2020-01           False     Cheryl Lee   
                       2020-01           False    Dana Robles   
                       2020-01           False  Jeanette Luna   
                       2020-01           False  Jeanette Luna   

                                                    email  \
region        category Month                                
North America Clothing 2020-01  sarahgonzales@example.com   
                       2020-01   schwartzlisa@example.org   
                       2020-01   hannahcurtis@example.org   
                       2020-01         gina90@example.org   
                       2020-01         gina90@example.org   

                                                phone  country  \
region        category Month                                

In [64]:
print("\n--- Unstacking and Restacking levels to reshape the data ---")

# Unstack 'Product_Category' to make it columns
# This moves the 'Product_Category' level from the index to the columns
try:
    sales_unstacked_category = transactions_multi_index.unstack(level=['region', 'category', 'Month'])
    sales_unstacked = transactions_multi_index.unstack(level=['region', 'category', 'Month'])
    print("\nDataFrame after unstacking 'Product_Category':")
    sales_unstacked_category.head()
except ValueError as e:
    print(f"ValueError: {e}")
    print("\nExplanation: `unstack` expects unique values at the level it's trying to unstack for each combination of the higher levels. Since there are two entries for ('North', 'Electronics') on '2024-01-05', pandas doesn't know which 'Sales_Amount' or 'Product_Name' to place in the new '2024-01-05' column, hence the error.")



--- Unstacking and Restacking levels to reshape the data ---
ValueError: Length mismatch: Expected axis has 150805 elements, new values have 1440 elements

Explanation: `unstack` expects unique values at the level it's trying to unstack for each combination of the higher levels. Since there are two entries for ('North', 'Electronics') on '2024-01-05', pandas doesn't know which 'Sales_Amount' or 'Product_Name' to place in the new '2024-01-05' column, hence the error.


In [65]:
# When you have duplicates at the level you want to unstack (or pivot),
# you should use `pivot_table` to aggregate the values.
# Here, we want to sum the Sales_Amount for duplicate entries.
sales_aggregated_pivot_table = transactions_multi_index.pivot_table(
    index=['region', 'category'], # These will form the new multi-index rows
    columns='date',                               # This will become the new columns
    values='total_amount',                        # The values to aggregate
    aggfunc='sum'                                 # How to handle duplicates (sum them up)
)
print("\nSales data aggregated and unstacked using `pivot_table` (handling duplicates):")
print(sales_aggregated_pivot_table)


Sales data aggregated and unstacked using `pivot_table` (handling duplicates):
date                       2020-01-01  2020-01-02  2020-01-03  2020-01-04  \
region        category                                                      
Africa        Beauty          4374.39     2939.24     8149.46    10484.81   
              Clothing        3370.30     1910.14     9278.20     5335.47   
              Electronics         NaN     2485.78    10311.61     1493.87   
              Food            1331.88     5010.39     3635.36     8071.89   
              Home                NaN     4700.02     6604.78     9163.88   
Asia          Beauty          3534.42    10699.18     5215.34     3734.47   
              Clothing        3013.47     9266.33     2116.36     9796.93   
              Electronics     1812.52     4186.20     2486.47     2642.96   
              Food             296.75    16228.81         NaN    16627.34   
              Home            2199.99     7805.27     1506.63     1382.53

3. **Advanced GroupBy Operations**
   - Use the `transform` method to normalize sales values within groups
   - Apply multiple aggregation functions simultaneously using `agg`
   - Implement custom aggregation functions
   - Use filter operations to select groups meeting specific criteria

In [68]:
# Use the `transform` method to normalize sales values within groups
# Normalize Sales_Amount within each Product_Category
# This calculates the z-score (standard score) for each sales amount relative to its product category.
# It returns a Series with the same index as the original DataFrame.
transactions_product_customer_df_merged['Sales_Amount_Normalized_by_Category'] = transactions_product_customer_df_merged.groupby('category')['total_amount'].transform(lambda x: (x - x.mean()) / x.std())
print("\nSales DataFrame with Sales_Amount normalized within each Product_Category (using transform):")
print(transactions_product_customer_df_merged[['category', 'total_amount', 'Sales_Amount_Normalized_by_Category']])



Sales DataFrame with Sales_Amount normalized within each Product_Category (using transform):
           category  total_amount  Sales_Amount_Normalized_by_Category
0              Food       1568.32                             0.286634
1              Food       2129.67                             0.854352
2          Clothing       3767.68                             2.299147
3              Food       2986.41                             1.720810
4            Beauty       2120.16                             1.089722
...             ...           ...                                  ...
150800       Beauty        234.71                            -0.903191
150801  Electronics        294.24                            -0.909390
150802     Clothing       1588.04                             0.093019
150803         Home        475.29                            -0.890065
150804     Clothing        377.42                            -1.132314

[150805 rows x 3 columns]


In [70]:
# Apply multiple aggregation functions simultaneously using `agg`
# Group by Customer_Region and Product_Category, then calculate mean, sum, and count of Sales_Amount
region_category_sales_summary = transactions_product_customer_df_merged.groupby(['region', 'category']).agg(
    Total_Sales=('total_amount', 'sum'),
    Average_Sales=('total_amount', 'mean'),
    Number_of_Sales=('total_amount', 'count'),
    Min_Sales=('total_amount', 'min'),
    Max_Sales=('total_amount', 'max')
)
print("\nSales Summary by Customer_Region and Product_Category (using agg with multiple functions):")
print(region_category_sales_summary)


Sales Summary by Customer_Region and Product_Category (using agg with multiple functions):
                           Total_Sales  Average_Sales  Number_of_Sales  \
region        category                                                   
Africa        Beauty        5932172.45    1068.090106             5554   
              Clothing      7297847.29    1488.445297             4903   
              Electronics   5423543.08    1146.626444             4730   
              Food          7876309.30    1281.951383             6144   
              Home          6555599.84    1350.556209             4854   
Asia          Beauty        6111817.95    1087.318618             5621   
              Clothing      7395392.89    1498.559856             4935   
              Electronics   5555530.98    1150.928316             4827   
              Food          7969754.41    1280.281833             6225   
              Home          6644991.14    1354.186089             4907   
Europe        Beauty

In [71]:
# Implement custom aggregation functions
# Let's create a custom function to calculate the Interquartile Range (IQR)
def iqr(series):
    return series.quantile(0.75) - series.quantile(0.25)

In [73]:
custom_agg_sales = transactions_product_customer_df_merged.groupby('category').agg(
    Total_Sales=('total_amount', 'sum'),
    IQR_Sales=('total_amount', iqr), # Using the custom IQR function
    Unique_Products=('product_id', lambda x: x.nunique()) # Custom lambda for unique count
)
print("\nSales Summary by Product_Category with custom aggregation (IQR and Unique Products):")
print(custom_agg_sales)


Sales Summary by Product_Category with custom aggregation (IQR and Unique Products):
             Total_Sales  IQR_Sales  Unique_Products
category                                            
Beauty       35023185.90    1356.18               43
Clothing     41749740.21    1450.89               37
Electronics  31298193.49    1230.83               36
Food         45547184.78    1443.09               47
Home         37724534.26    1459.43               37


In [104]:
# Use filter operations to select groups meeting specific criteria
# Filter groups where the total sales for a Product_Category is greater than 5000000
high_sales_categories = transactions_product_customer_df_merged.groupby('category').filter(lambda x: x['total_amount'].sum() > 5000).groupby('category')
print("\nSales records for Product_Categories with total sales > 5000000 (using filter):")
print(high_sales_categories)
# print(high_sales_categories[['category', 'Total_Sales']])



Sales records for Product_Categories with total sales > 5000000 (using filter):


In [109]:
summary_high_sales = high_sales_categories.agg(
    Total_Sales=('total_amount', 'sum'),
    Average_Sales=('total_amount', 'mean'),
    Number_of_Sales=('total_amount', 'count'),
    Min_Sales=('total_amount', 'min'),
    Max_Sales=('total_amount', 'max')
)
print("\nSummary of high sales categories:")
print(summary_high_sales)


Summary of high sales categories:
             Total_Sales  Average_Sales  Number_of_Sales  Min_Sales  Max_Sales
category                                                                      
Beauty       35023185.90    1089.198753            32155       -1.0    3848.00
Clothing     41749740.21    1496.138334            27905       -1.0    3968.84
Electronics  31298193.49    1144.483618            27347       -1.0    3936.76
Food         45547184.78    1284.901399            35448       -1.0    3981.88
Home         37724534.26    1349.715000            27950       -1.0    3982.72


In [111]:
# Filter groups where a Customer_Region has more than 3 sales records
active_regions = transactions_product_customer_df_merged.groupby('region').filter(lambda x: len(x) > 3).groupby('region')
print("\nSales records for Customer_Regions with more than 3 sales (using filter):")
print(active_regions)


Sales records for Customer_Regions with more than 3 sales (using filter):


In [112]:
summary_active_regions = active_regions.agg(
    Total_Sales=('total_amount', 'sum'),
    Average_Sales=('total_amount', 'mean'),
    Number_of_Sales=('total_amount', 'count'),
    Min_Sales=('total_amount', 'min'),
    Max_Sales=('total_amount', 'max')
)
print("\nSummary of active regions:")
print(summary_active_regions)


Summary of active regions:
               Total_Sales  Average_Sales  Number_of_Sales  Min_Sales  \
region                                                                  
Africa         33085471.96    1263.527667            26185       -1.0   
Asia           33677487.37    1270.129639            26515       -1.0   
Europe         30525134.50    1266.287833            24106       -1.0   
North America  26718240.29    1254.907721            21291       -1.0   
Oceania        33604073.44    1282.304565            26206       -1.0   
South America  33732431.08    1272.825865            26502       -1.0   

               Max_Sales  
region                    
Africa           3982.72  
Asia             3982.72  
Europe           3982.72  
North America    3982.72  
Oceania          3982.72  
South America    3982.72  


### Task 2: Advanced Merging and Joining

1. **Complex Joins**

   - Perform a three-way join between sales, customer, and product datasets
   - Implement a self-join on the customer dataset to identify hierarchical relationships
   - Use different join types (left, right, inner, outer) and compare the results